# CCF-opt mask via relaxed seed + inject-bias cull

Build a new packaged ``*.ccfopt.delta.mas`` from the **relaxed** depth>=10% seed
masks (no deblend / telluric / saturated-core cuts) using the **same** product
tree as ``rv_order_scan_sim.ipynb``:

``OUT = /Users/ckilpatrick/hispec_rv_order_scan``

Workflow:

1. Reuse (or rebuild) shared Stage-1 cals, raw UTR ramps, and Stage0-2 blaze L2
   products under ``OUT`` -- defaults **skip** any step whose outputs already exist.
2. At inject = 0: line-by-line CCF on the relaxed ``*.delta.mas`` seed -> re-zero
   each line (``lambda' = lambda * (1 + RV/c)``) and set weight proportional to
   ``|Gaussian contrast|``.
3. With that interim rezeroed mask, run line-by-line stellar RV at inject =
   0, 100, ..., 1000 m/s (writes under a distinct ``stellar_rv_biascull`` suffix so
   existing ``stellar_rv`` products are not clobbered).
4. Per line: average bias ``<RV - inject>`` over the inject grid -> histogram;
   drop extreme outliers with a **tunable** ``BIAS_ABS_CUT_M_S``.
5. Write the culled list to ``hispecdrp/data/masks/<stem>.ccfopt.delta.mas``.

After changing the mask or CCF code, set ``RERUN_BIAS_CULL_RV=True`` (and/or
``REBUILD_REZERO_MASK=True``) and re-run from the relevant section -- you do **not**
need to re-simulate or re-reduce Stages 0-2.


In [ ]:
import os
import hispecdrp

print(f"hispecdrp {hispecdrp.__version__} @ {hispecdrp.__file__}")

# Exact same product tree as rv_order_scan_sim.ipynb
OUT = "/Users/ckilpatrick/hispec_rv_order_scan"
os.makedirs(OUT, exist_ok=True)

# Analysis / intermediate products (do not overwrite per_order_rv_scan.*)
ANALYSIS_DIR = os.path.join(OUT, "ccfopt_bias_cull")
os.makedirs(ANALYSIS_DIR, exist_ok=True)

os.environ.setdefault(
    "KOA_CALIBRATION_CACHE",
    os.path.join(OUT, "cals", "KOA_CALIBRATION_CACHE"),
)
print("output:", OUT)
print("analysis:", ANALYSIS_DIR)
print("KOA_CALIBRATION_CACHE:", os.environ["KOA_CALIBRATION_CACHE"])


## 1. Sync / verify KOA caldb (BSPEC)


In [ ]:
from hispecdrp.calibrations import verify_koa_calibrations

SPECTROGRAPH = "BSPEC"
summary = verify_koa_calibrations(SPECTROGRAPH, ignore_missing_cal_types=["lfc_wavesol"])

print(
    f"{summary['spectrograph']} caldb: {summary['n_records']} record(s); "
    f"downloaded {summary['n_downloaded']}/{summary['n_missing']} missing file(s)"
)
print("types:", summary["cal_types"])
if summary["downloaded"]:
    print("downloaded:", summary["downloaded"])
if summary["failed"]:
    print("failed:", summary["failed"])


## 2. Injection / mask / cache configuration

Uses the packaged **relaxed** ``*.delta.mas`` seed (depth >= 10%; no blend /
telluric filters). Cache flags default to reuse of existing OUT products.

| Flag | Default | Effect |
|---|---|---|
| `RESIMULATE_RAW` | `False` | Re-run 1D->ZEMAX->UTR only if True or ramp missing |
| `RERUN_PIPELINE` | `False` | Re-run Stage0-2 (through blaze, **no** stellar RV) only if True or blaze L2 missing |
| `REBUILD_SHARED_CALS` | `False` | Rebuild shared Stage-1 bundle only if True or missing |
| `REBUILD_REZERO_MASK` | `False` | Rebuild interim rezeroed mask from inj=0 only if True or missing |
| `RERUN_BIAS_CULL_RV` | `False` | Re-run bias-cull stellar RV on all injects only if True or product missing |
| `USE_SINGLE_LINE_BUILDER` | `False` | If True, rezero via per-line single-line CCF (slow, gold-standard); else one full-mask line-by-line RVCCF + `build_ccfopt_line_mask_from_rvccf_table` |
| `BIAS_ABS_CUT_M_S` | `5.0` | Drop lines with `|mean bias| >=` this (m/s); inspect histogram first and retune |

Stellar-RV products for this notebook use suffix ``stellar_rv_biascull`` so they
do not clobber ``rv_order_scan_sim``'s ``stellar_rv`` files.


In [ ]:
from pathlib import Path

import numpy as np

from hispecdrp import datamodels
from hispecdrp.utils.line_mask_utils import (
    CCFOPT_SINGLE_LINE_STELLAR_RV_KW,
    package_data_masks_dir,
)

FIBERS = [2]
FIBER = FIBERS[0]
INJECT_RV_VIA = "spectrum"
EXTRACTION_APERTURE = 8
TEMPLATE = "lte049.0-2.5-0.0a+0.0.BT-Settl.spec.7.h5"
TEMPLATE_STEM = Path(TEMPLATE).stem

MASKS_DIR = package_data_masks_dir()
SEED_MASK = MASKS_DIR / f"{TEMPLATE_STEM}.delta.mas"
OUT_CCFOPT = MASKS_DIR / f"{TEMPLATE_STEM}.ccfopt.delta.mas"
INTERIM_REZERO = Path(ANALYSIS_DIR) / f"{TEMPLATE_STEM}.ccfopt.rezeroed.delta.mas"
BIAS_TABLE_PATH = Path(ANALYSIS_DIR) / "per_line_inject_bias.ecsv"
BIAS_HIST_PNG = Path(ANALYSIS_DIR) / "per_line_mean_bias_hist.png"
STELLAR_RV_SUFFIX = "stellar_rv_biascull"

POISSON_NOISE = False
CONVERT_TO_UINT16 = False
SEED = 0

# Cache / rerun controls -- leave False unless you intentionally want a rebuild.
RESIMULATE_RAW = False
RERUN_PIPELINE = False
REBUILD_SHARED_CALS = False
REBUILD_REZERO_MASK = False
RERUN_BIAS_CULL_RV = False

# Rezero builder: False = fast (one full-mask line-by-line RVCCF at inj=0).
USE_SINGLE_LINE_BUILDER = False
# Optional clip of seed lines to L2 wavelength coverage (speeds rezero).
CLIP_SEED_TO_L2_WAVE = True
SEED_WAVE_MARGIN_UM = 0.002

# Tunable cull after inspecting the histogram in section 8.
BIAS_ABS_CUT_M_S = 5.0
WRITE_PACKAGED_CCFOPT = True

# 0 -> 1000 m/s in 100 m/s steps (11 runs)
INJECT_M_S = np.arange(0.0, 1000.0 + 0.5, 100.0)
assert len(INJECT_M_S) == 11

assert SEED_MASK.is_file(), f"missing relaxed seed mask: {SEED_MASK}"


def inj_label(inject_m_s: float) -> str:
    return f"inj_{int(round(float(inject_m_s))):05d}mps"


def inj_dir(inject_m_s: float) -> Path:
    return Path(OUT) / inj_label(inject_m_s)


def ramp_path(inject_m_s: float) -> Path:
    return inj_dir(inject_m_s) / "simulated_ramp.fits"


def find_l2_blaze_path(run_dir: Path | str) -> Path | None:
    # Latest blaze-corrected L2 (HB.*_spec*.fits, not uncal / not stellar_rv*).
    run_dir = Path(run_dir)
    cands = [
        p
        for p in sorted(run_dir.glob("HB.*_spec*.fits"))
        if "uncal" not in p.name and "stellar_rv" not in p.name
    ]
    return cands[-1] if cands else None


def find_biascull_l2_path(run_dir: Path | str) -> Path | None:
    run_dir = Path(run_dir)
    hits = sorted(run_dir.glob(f"HB.*_spec-{STELLAR_RV_SUFFIX}*.fits"))
    return hits[-1] if hits else None


def l2_has_rvccf(path: Path | str | None) -> bool:
    if path is None or not Path(path).is_file():
        return False
    try:
        m = datamodels.open(str(path))
    except Exception:
        return False
    tbl = getattr(m, f"rvccf_fib{FIBER}", None)
    return tbl is not None and len(tbl) > 0


def stellar_rv_primitives(
    ccf_input_velocity_m_s: float,
    *,
    mask_path: str,
    save_preview: bool = False,
) -> dict:
    kw = CCFOPT_SINGLE_LINE_STELLAR_RV_KW
    return {
        "algorithm": "ccf_line_mask",
        "mask_path": str(mask_path),
        "ccf_input_velocity": float(ccf_input_velocity_m_s),
        "do_fibers": list(FIBERS),
        "window_line_by_line": bool(kw["window_line_by_line"]),
        "window_size_um": float(kw["window_size_um"]),
        "v_half_window": float(kw["v_half_window_m_s"]),
        "v_step_coarse": float(kw["v_step_m_s"]),
        "mask_ccf_pixel_interp_order": int(kw["mask_ccf_pixel_interp_order"]),
        "ccf_zb": 0.0,
        "save_preview": bool(save_preview),
        "preview_write_png": False,
        "show_preview": False,
    }


print(f"fiber(s): {FIBERS}")
print(f"extraction_aperture = {EXTRACTION_APERTURE}")
print(f"inject_rv_via = {INJECT_RV_VIA!r}")
print(f"seed mask (relaxed): {SEED_MASK}")
print(f"interim rezero: {INTERIM_REZERO}")
print(f"packaged ccfopt out: {OUT_CCFOPT}")
print(f"poisson_noise={POISSON_NOISE}  convert_to_uint16={CONVERT_TO_UINT16}")
print(f"seed={SEED}")
print(
    f"RESIMULATE_RAW={RESIMULATE_RAW}  RERUN_PIPELINE={RERUN_PIPELINE}  "
    f"REBUILD_SHARED_CALS={REBUILD_SHARED_CALS}"
)
print(
    f"REBUILD_REZERO_MASK={REBUILD_REZERO_MASK}  RERUN_BIAS_CULL_RV={RERUN_BIAS_CULL_RV}  "
    f"USE_SINGLE_LINE_BUILDER={USE_SINGLE_LINE_BUILDER}"
)
print(f"BIAS_ABS_CUT_M_S={BIAS_ABS_CUT_M_S}")
print(f"inject steps (m/s): {INJECT_M_S.tolist()}")


## 3. Shared Stage-1 cals

Same as ``rv_order_scan_sim`` -- reused from ``OUT/_shared_stage1_rv0`` when present.


In [ ]:
from hispecdrp.simulations.e2e_pipeline import build_wavesol_aidan_with_order_zp
from hispecdrp.simulations.simulate_up_the_ramp import (
    create_ramp_from_spectrum,
    get_expected_ramp_path,
    load_cached_sim_stage1_masters,
    sim_stage1_masters_exist,
)

_shared_dir = str(Path(OUT) / "_shared_stage1_rv0")
os.makedirs(_shared_dir, exist_ok=True)

if (
    not REBUILD_SHARED_CALS
    and sim_stage1_masters_exist(
        _shared_dir,
        SPECTROGRAPH,
        FIBERS,
        extraction_aperture=float(EXTRACTION_APERTURE),
    )
    and Path(_shared_dir, "rate_map.fits").is_file()
):
    print(f"Reusing shared Stage-1 cals under {_shared_dir}/cals ...")
    _bundle = load_cached_sim_stage1_masters(_shared_dir, SPECTROGRAPH)
    SHARED_TRACEPOS = _bundle["trace_pos"]
    SHARED_TRACEPROF = _bundle["trace_prof"]
    SHARED_FIBERFLAT = _bundle["fiberflat"]
    from astropy.io import fits as _fits

    RATE_MAP_REF = np.asarray(
        _fits.getdata(str(Path(_shared_dir) / "rate_map.fits")), dtype=np.float64
    )
    SHARED_WAVESOL, ZP_INFO = build_wavesol_aidan_with_order_zp(
        RATE_MAP_REF,
        spectrograph=SPECTROGRAPH,
        fibers=FIBERS,
        known_rv_m_s=0.0,
    )
else:
    print(
        "Building shared Stage-1 cals (RV=0 -> fiberflat/traces aperture="
        f"{EXTRACTION_APERTURE}, Aidan gws + order ZP0) ..."
    )
    _ramp0, _cals0, RATE_MAP_REF = create_ramp_from_spectrum(
        _shared_dir,
        spectrograph=SPECTROGRAPH,
        fibers=FIBERS,
        template_filename=TEMPLATE,
        mag_ab=3.05,
        filter_band="J",
        seed=SEED,
        radial_velocity_km_s=0.0,
        barycentric_velocity_km_s=0.0,
        inject_rv_via=INJECT_RV_VIA,
        poisson_noise=POISSON_NOISE,
        convert_to_uint16=CONVERT_TO_UINT16,
        extraction_aperture=float(EXTRACTION_APERTURE),
        reuse_stage1_sims=not REBUILD_SHARED_CALS,
    )
    SHARED_TRACEPOS = _cals0["trace_pos"]
    SHARED_TRACEPROF = _cals0["trace_prof"]
    SHARED_FIBERFLAT = _cals0["fiberflat"]
    SHARED_WAVESOL, ZP_INFO = build_wavesol_aidan_with_order_zp(
        RATE_MAP_REF,
        spectrograph=SPECTROGRAPH,
        fibers=FIBERS,
        known_rv_m_s=0.0,
        meta={"mjd_start": getattr(getattr(_ramp0, "meta", None), "mjd_start", 0.0)},
    )

print(
    f"SHARED_WAVESOL note={getattr(SHARED_WAVESOL.meta, 'note', None)}  "
    f"median order ZP = {ZP_INFO.get('median_dv_m_s', float('nan')):.3f} m/s  "
    f"(poly_deg={ZP_INFO.get('poly_deg')})"
)


## 4. Simulate all raw ramps (1D -> Doppler -> ZEMAX -> UTR)

Default ``RESIMULATE_RAW=False`` skips steps that already have a ramp on disk.


In [ ]:
RAW_MANIFEST = []
for inj in INJECT_M_S:
    label = inj_label(inj)
    run_dir = inj_dir(inj)
    run_dir.mkdir(parents=True, exist_ok=True)
    rpath = ramp_path(inj)
    if (not RESIMULATE_RAW) and rpath.is_file():
        print(f"[raw skip] {label}: reuse {rpath}")
        RAW_MANIFEST.append(
            {"label": label, "inject_m_s": float(inj), "ramp_path": str(rpath), "reused": True}
        )
        continue

    print(f"\n=== RAW {label}: inject {inj:.1f} m/s  (1D->ZEMAX->UTR) ===")
    ramp, _cals, rate_map = create_ramp_from_spectrum(
        str(run_dir),
        spectrograph=SPECTROGRAPH,
        fibers=FIBERS,
        template_filename=TEMPLATE,
        mag_ab=3.05,
        filter_band="J",
        seed=SEED,
        radial_velocity_km_s=float(inj) / 1000.0,
        barycentric_velocity_km_s=0.0,
        inject_rv_via=INJECT_RV_VIA,
        poisson_noise=POISSON_NOISE,
        convert_to_uint16=CONVERT_TO_UINT16,
        register_stage1_sims=False,
        overwrite_ramp=True,
    )
    print(f"  wrote {get_expected_ramp_path(str(run_dir))}  shape={ramp.data.shape}")
    RAW_MANIFEST.append(
        {
            "label": label,
            "inject_m_s": float(inj),
            "ramp_path": get_expected_ramp_path(str(run_dir)),
            "reused": False,
        }
    )

n_reuse = sum(1 for r in RAW_MANIFEST if r["reused"])
print(
    f"\nRaw sim done: {len(RAW_MANIFEST)} steps  "
    f"({n_reuse} reused, {len(RAW_MANIFEST) - n_reuse} rebuilt)"
)


## 5. Stage0-2 pipeline on cached ramps (through blaze; **no** stellar RV)

Default ``RERUN_PIPELINE=False`` reuses existing blaze L2 products.


In [ ]:
from hispecdrp.pipelines import Stage0Pipeline, Stage1Pipeline, Stage2Pipeline


def run_pipeline_through_blaze(inject_m_s: float) -> dict:
    # Stage0-2 through blaze (stellar_rv skipped) on cached ramp.
    label = inj_label(inject_m_s)
    run_dir = inj_dir(inject_m_s)
    injected_m_s = float(inject_m_s)
    rpath = ramp_path(inject_m_s)
    if not rpath.is_file():
        raise FileNotFoundError(
            f"Missing raw ramp for {label}: {rpath}. Run section 4 with RESIMULATE_RAW=True."
        )

    existing = find_l2_blaze_path(run_dir)
    if (not RERUN_PIPELINE) and existing is not None and existing.is_file():
        print(f"[pipe skip] {label}: reuse blaze L2 {existing}")
        return {
            "label": label,
            "run_dir": str(run_dir),
            "l2_blaze_path": str(existing),
            "inject_m_s": injected_m_s,
        }

    print(f"\n=== PIPE {label}: inject {injected_m_s:.1f} m/s  Stage0-2 (no stellar RV) ===")
    ramp = datamodels.open(str(rpath))
    do_fibers = [str(f) for f in FIBERS]
    l0 = Stage0Pipeline.call(ramp, output_dir=str(run_dir))
    l1 = Stage1Pipeline(
        output_dir=str(run_dir),
        primitives={
            "spec_extract": {
                "do_fibers": do_fibers,
                "method": "box",
                "extraction_aperture": int(EXTRACTION_APERTURE),
                "tracepos": SHARED_TRACEPOS,
                "traceprof": SHARED_TRACEPROF,
            },
            "set_wavesol": {"wavesol": SHARED_WAVESOL},
        },
    ).apply(l0)
    l2 = Stage2Pipeline(
        output_dir=str(run_dir),
        primitives={
            "blaze_corr": {
                "do_fibers": do_fibers,
                "fiberflat": SHARED_FIBERFLAT,
            },
            "sky_sub": {"run": False},
            "barycorr": {"run": False},
            "stellar_rv": {"run": False},
        },
    ).apply(l1)
    l2_path = getattr(l2.meta, "filename", None) or str(find_l2_blaze_path(run_dir))
    save_as = run_dir / Path(l2_path).name if l2_path else run_dir / f"{label}_spec.fits"
    if not Path(l2_path).is_file():
        l2.save(filepath=str(save_as))
        l2_path = str(save_as)
    print(f"  blaze L2 saved: {l2_path}")
    return {
        "label": label,
        "run_dir": str(run_dir),
        "l2_blaze_path": str(l2_path),
        "inject_m_s": injected_m_s,
    }


PIPE_RESULTS = []
for inj in INJECT_M_S:
    PIPE_RESULTS.append(run_pipeline_through_blaze(inj))

print(f"\nBlaze L2 products ({len(PIPE_RESULTS)}):")
for r in PIPE_RESULTS:
    print(f"  {r['label']}: {r['l2_blaze_path']}")


## 6. Rezero + contrast weights at inject = 0 (relaxed seed -> interim ccfopt)

Uses the blaze L2 under ``inj_00000mps/``. Writes
``OUT/ccfopt_bias_cull/<stem>.ccfopt.rezeroed.delta.mas``.

- ``USE_SINGLE_LINE_BUILDER=False`` (default): one full-mask line-by-line stellar RV,
  then ``build_ccfopt_line_mask_from_rvccf_table``.
- ``True``: ``build_ccfopt_line_mask_from_l2_single_line_ccf`` (slow; gold-standard).

Set ``REBUILD_REZERO_MASK=True`` to force a rebuild after changing the seed / CCF code.


In [ ]:
from hispecdrp.primitives.stellar_rv.stellar_rv_primitive import StellarRVPrimitive
from hispecdrp.utils.line_mask_utils import (
    MaskLines,
    build_ccfopt_line_mask_from_l2_single_line_ccf,
    build_ccfopt_line_mask_from_rvccf_table,
    read_mask_generation_meta,
)

tpl_path = Path(hispecdrp.__file__).resolve().parent / "data" / "templates" / TEMPLATE
inj0_blaze = find_l2_blaze_path(inj_dir(0.0))
if inj0_blaze is None or not inj0_blaze.is_file():
    raise FileNotFoundError("Missing inj=0 blaze L2; run section 5 first.")

l2_inj0 = datamodels.open(str(inj0_blaze))
wave0 = np.asarray(getattr(l2_inj0, f"wave_fib{FIBER}"), dtype=np.float64)
wmin = float(np.nanmin(wave0)) - float(SEED_WAVE_MARGIN_UM)
wmax = float(np.nanmax(wave0)) + float(SEED_WAVE_MARGIN_UM)
print(f"inj0 blaze L2: {inj0_blaze}")
print(f"wave_fib{FIBER} range: [{np.nanmin(wave0):.4f}, {np.nanmax(wave0):.4f}] um")

seed_for_rezero = Path(SEED_MASK)
if CLIP_SEED_TO_L2_WAVE:
    seed_ml = MaskLines.load(str(SEED_MASK))
    lam = np.asarray(seed_ml.line_lambda_um, dtype=np.float64)
    w = np.asarray(seed_ml.weight, dtype=np.float64)
    ok = (
        np.isfinite(lam)
        & (lam > 0)
        & np.isfinite(w)
        & (w > 0)
        & (lam >= wmin)
        & (lam <= wmax)
    )
    clipped = Path(ANALYSIS_DIR) / f"{TEMPLATE_STEM}.delta.bspec_clip.mas"
    if int(ok.sum()) == 0:
        raise RuntimeError("CLIP_SEED_TO_L2_WAVE left zero seed lines")
    MaskLines.from_delta_centers_um(lam[ok], w[ok] / np.sum(w[ok])).save(
        output_path=str(clipped)
    )
    seed_for_rezero = clipped
    print(
        f"Clipped seed to L2 wave +/- {SEED_WAVE_MARGIN_UM} um: "
        f"{int(ok.sum())} lines -> {clipped}"
    )
else:
    print(f"Using full relaxed seed: {seed_for_rezero}")

if (not REBUILD_REZERO_MASK) and INTERIM_REZERO.is_file():
    print(f"[rezero skip] reuse interim mask {INTERIM_REZERO}")
    REZERO_MASK_PATH = str(INTERIM_REZERO)
    _meta = read_mask_generation_meta(REZERO_MASK_PATH) or {}
    print(f"  n_lines={_meta.get('n_lines')}")
else:
    print(
        f"Building interim rezeroed mask "
        f"(USE_SINGLE_LINE_BUILDER={USE_SINGLE_LINE_BUILDER}) ..."
    )
    if USE_SINGLE_LINE_BUILDER:
        REZERO_MASK_PATH, entries = build_ccfopt_line_mask_from_l2_single_line_ccf(
            str(seed_for_rezero),
            l2_inj0,
            fiber=FIBER,
            template_path=str(tpl_path),
            output_path=str(INTERIM_REZERO),
            masks_directory=str(ANALYSIS_DIR),
        )
    else:
        # One full-mask line-by-line RV at inject=0, then rezero+weight from RVCCF.
        _rezero_l2_path = Path(ANALYSIS_DIR) / "inj0_seed_line_by_line_stellar_rv.fits"
        if (
            (not REBUILD_REZERO_MASK)
            and _rezero_l2_path.is_file()
            and l2_has_rvccf(_rezero_l2_path)
        ):
            print(f"  reuse seed RVCCF L2 {_rezero_l2_path}")
            l2_seed_rv = datamodels.open(str(_rezero_l2_path))
        else:
            prim0 = StellarRVPrimitive(
                **stellar_rv_primitives(
                    0.0, mask_path=str(seed_for_rezero), save_preview=False
                ),
                save_result=True,
                output_dir=str(ANALYSIS_DIR),
                suffix="seed_line_by_line",
            )
            l2_seed_rv = prim0.apply(l2_inj0)
            outp = getattr(l2_seed_rv.meta, "filename", None)
            if outp and Path(outp).is_file():
                import shutil

                shutil.copy2(outp, _rezero_l2_path)
            else:
                l2_seed_rv.save(filepath=str(_rezero_l2_path))
        tbl0 = getattr(l2_seed_rv, f"rvccf_fib{FIBER}", None)
        if tbl0 is None or len(tbl0) == 0:
            raise RuntimeError("seed line-by-line RV produced empty RVCCF")
        print(f"  seed RVCCF rows: {len(tbl0)}")
        REZERO_MASK_PATH, entries = build_ccfopt_line_mask_from_rvccf_table(
            str(seed_for_rezero),
            tbl0,
            template_path=str(tpl_path),
            output_path=str(INTERIM_REZERO),
            masks_directory=str(ANALYSIS_DIR),
        )
    meta = read_mask_generation_meta(REZERO_MASK_PATH) or {}
    print(f"wrote {len(entries)} rezeroed lines -> {REZERO_MASK_PATH}")
    print(
        f"  n_seed={meta.get('n_seed_lines')}  n_drop={meta.get('n_dropped_lines')}  "
        f"weight_def={meta.get('mask_weight_definition')}"
    )

ml_rz = MaskLines.load(REZERO_MASK_PATH)
print(
    f"interim mask weight sum = {float(np.sum(ml_rz.weight)):.9f}  "
    f"n_positive = {int(np.sum(np.asarray(ml_rz.weight) > 0))}"
)


## 7. Line-by-line stellar RV at all injects (interim rezeroed mask)

Writes ``HB.*_spec-stellar_rv_biascull*.fits`` under each ``inj_*`` directory.
Default ``RERUN_BIAS_CULL_RV=False`` reuses existing bias-cull RV products.


In [ ]:
def run_biascull_stellar_rv(inject_m_s: float, *, save_preview: bool = False) -> dict:
    label = inj_label(inject_m_s)
    run_dir = inj_dir(inject_m_s)
    injected_m_s = float(inject_m_s)

    blaze = find_l2_blaze_path(run_dir)
    if blaze is None or not blaze.is_file():
        raise FileNotFoundError(
            f"Missing blaze L2 for {label} under {run_dir}. Run section 5 first."
        )

    existing_rv = find_biascull_l2_path(run_dir)
    if (not RERUN_BIAS_CULL_RV) and l2_has_rvccf(existing_rv):
        print(f"[rv skip] {label}: reuse {existing_rv}")
        l2 = datamodels.open(str(existing_rv))
        l2_path = str(existing_rv)
    else:
        print(
            f"\n=== BIAS-CULL RV {label}: inject {injected_m_s:.1f} m/s  "
            f"mask={Path(REZERO_MASK_PATH).name} ==="
        )
        prim = StellarRVPrimitive(
            **stellar_rv_primitives(
                injected_m_s,
                mask_path=REZERO_MASK_PATH,
                save_preview=bool(save_preview),
            ),
            save_result=True,
            output_dir=str(run_dir),
            suffix=STELLAR_RV_SUFFIX,
        )
        l2 = prim.apply(str(blaze))
        l2_path = getattr(l2.meta, "filename", None) or str(
            find_biascull_l2_path(run_dir) or blaze
        )
        if not Path(l2_path).is_file():
            save_as = run_dir / f"{Path(blaze).stem}-{STELLAR_RV_SUFFIX}.fits"
            l2.save(filepath=str(save_as))
            l2_path = str(save_as)
        print(f"  L2+RV saved: {l2_path}")

    tbl = getattr(l2, f"rvccf_fib{FIBER}", None)
    if tbl is None or len(tbl) == 0:
        raise RuntimeError(f"{label}: missing rvccf_fib{FIBER} in {l2_path}")
    avg = getattr(l2, f"rvccfavg_fib{FIBER}", None)
    rv_rec = float(avg["RV"][0]) if avg is not None and len(avg) else float("nan")
    rv_err = float(avg["RV_ERR"][0]) if avg is not None and len(avg) else float("nan")
    print(
        f"  fiber {FIBER}: agg RV = {rv_rec:.6f} +/- {rv_err:.6f} m/s  "
        f"bias = {rv_rec - injected_m_s:.6f} m/s  n_lines={len(tbl)}"
    )
    return {
        "label": label,
        "run_dir": str(run_dir),
        "l2_path": str(l2_path),
        "inject_m_s": injected_m_s,
        "recover_m_s": rv_rec,
        "recover_err_m_s": rv_err,
        "rvccf": tbl,
    }


BIAS_RV_RESULTS = []
for inj in INJECT_M_S:
    BIAS_RV_RESULTS.append(run_biascull_stellar_rv(inj, save_preview=False))

print("\nAggregate recovery (interim rezeroed mask):")
for r in BIAS_RV_RESULTS:
    print(
        f"  inject={r['inject_m_s']:7.1f} m/s  "
        f"agg={r['recover_m_s']:+10.3f}  "
        f"bias={r['recover_m_s'] - r['inject_m_s']:+8.3f} m/s"
    )


## 8. Per-line mean bias histogram + tunable cull

For each mask line, track ``bias = RV - inject`` across the inject grid and take
the mean. Inspect the histogram, then set ``BIAS_ABS_CUT_M_S`` in section 2 and
re-run **this cell** (and section 9) to write the packaged ccfopt.

Uses `hispecdrp.simulations.plotting.analyze_line_bias_and_plot`.


In [ ]:
from hispecdrp.simulations.plotting import analyze_line_bias_and_plot

result = analyze_line_bias_and_plot(
    BIAS_RV_RESULTS,
    REZERO_MASK_PATH,
    bias_table_path=BIAS_TABLE_PATH,
    hist_png=BIAS_HIST_PNG,
    bias_abs_cut_m_s=BIAS_ABS_CUT_M_S,
    show=True,
)

LINE_BIAS = result["line_bias"]
ref_lam = result["ref_lam"]
ref_w = result["ref_w"]
mean_bias = result["mean_bias"]
rms_bias = result["rms_bias"]
finite = result["finite"]
keep = result["keep"]

print(f"Wrote {result['bias_table_path']}  ({len(LINE_BIAS)} lines)")
print(
    f"mean bias stats (all finite lines, n={result['n_finite']}): "
    f"median={result['median_m_s']:+.3f}  MAD={result['mad_m_s']:.3f}  "
    f"p95={result['p95_abs_m_s']:.3f}  "
    f"p99={result['p99_abs_m_s']:.3f}  "
    f"max|b|={result['max_abs_m_s']:.3f} m/s"
)
print(f"Current BIAS_ABS_CUT_M_S = {BIAS_ABS_CUT_M_S}")
print(f"Wrote {result['png']}")
print(
    f"Cull |mean_bias| >= {BIAS_ABS_CUT_M_S} m/s: "
    f"keep {result['n_keep']} / {result['n_finite']} finite lines  "
    f"(drop {result['n_drop']})"
)


## 9. Write packaged ``*.ccfopt.delta.mas``

Renormalizes kept-line weights to sum 1 and writes the packaged ccfopt product.
Re-run after adjusting ``BIAS_ABS_CUT_M_S``.


In [ ]:
from hispecdrp.utils.line_mask_utils import MaskLines, read_mask_generation_meta

keep = (
    np.isfinite(mean_bias)
    & (np.abs(mean_bias) < float(BIAS_ABS_CUT_M_S))
    & np.isfinite(ref_lam)
    & (ref_lam > 0)
    & np.isfinite(ref_w)
    & (ref_w > 0)
)
lam_keep = ref_lam[keep]
w_keep = ref_w[keep].copy()
w_keep /= float(np.sum(w_keep))

meta_src = read_mask_generation_meta(REZERO_MASK_PATH) or {}
meta = dict(meta_src)
meta.update(
    {
        "generator": (
            "hispecdrp.simulations.notebooks.ccfopt_bias_cull_from_order_scan"
        ),
        "seed_mask_path": str(Path(SEED_MASK).resolve()),
        "interim_rezero_mask_path": str(Path(REZERO_MASK_PATH).resolve()),
        "order_scan_out": str(OUT),
        "analysis_dir": str(ANALYSIS_DIR),
        "bias_abs_cut_m_s": float(BIAS_ABS_CUT_M_S),
        "n_lines_before_bias_cull": int(ref_lam.size),
        "n_lines_after_bias_cull": int(lam_keep.size),
        "n_lines_dropped_bias_cull": int(ref_lam.size - lam_keep.size),
        "inject_grid_m_s": [float(x) for x in INJECT_M_S],
        "mask_weight_definition": (
            "abs_gaussian_ccf_contrast_renormalized_after_bias_cull"
        ),
        "wavelength_offset_definition": (
            "lambda_prime = lambda_seed * (1 + RV/c) at inject=0"
        ),
        "weights_sum_normalized_to": 1.0,
        "n_lines": int(lam_keep.size),
    }
)

ml_out = MaskLines.from_delta_centers_um(lam_keep, w_keep, metadata=meta)
if WRITE_PACKAGED_CCFOPT:
    path_written = ml_out.save(output_path=str(OUT_CCFOPT))
    print(f"Wrote packaged ccfopt ({lam_keep.size} lines) -> {path_written}")
else:
    preview = Path(ANALYSIS_DIR) / f"{TEMPLATE_STEM}.ccfopt.delta.preview.mas"
    path_written = ml_out.save(output_path=str(preview))
    print(f"WRITE_PACKAGED_CCFOPT=False; wrote preview -> {path_written}")

# Also keep a copy under ANALYSIS_DIR.
_analysis_copy = Path(ANALYSIS_DIR) / Path(path_written).name
if Path(path_written).resolve() != _analysis_copy.resolve():
    import shutil

    shutil.copy2(path_written, _analysis_copy)
    print(f"Analysis copy: {_analysis_copy}")

print(
    f"weight sum = {float(np.sum(w_keep)):.9f}  "
    f"lambda range = [{lam_keep.min():.4f}, {lam_keep.max():.4f}] um"
)


## 10. Optional closed-loop check at inject = 0

Full-mask line-by-line stellar RV with the **new** packaged ccfopt on the inj=0
blaze L2. Expect aggregate bias ~ 0.


In [ ]:
from hispecdrp.primitives.stellar_rv.stellar_rv_primitive import StellarRVPrimitive

check_mask = str(OUT_CCFOPT if WRITE_PACKAGED_CCFOPT else path_written)
print(f"Closed-loop check with {check_mask}")
l2_chk = StellarRVPrimitive.call(
str(inj0_blaze),
    **stellar_rv_primitives(0.0, mask_path=check_mask, save_preview=True),
    save_result=True,
    output_dir=str(ANALYSIS_DIR),
    suffix="stellar_rv_ccfopt_check",
)
avg = getattr(l2_chk, f"rvccfavg_fib{FIBER}", None)
tbl = getattr(l2_chk, f"rvccf_fib{FIBER}", None)
if avg is None or len(avg) == 0:
    raise RuntimeError("missing RVCCFAVG after closed-loop check")
print(
    f"RVCCFAVG: {float(avg['RV'][0]):.6f} +/- {float(avg['RV_ERR'][0]):.6f} m/s  "
    f"(n_lines={len(tbl) if tbl is not None else 0})"
)
